**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
# Parametri Job
job_name = "train_s1_v3"                                    # nome da passare poi a notebook moco   train_s2_v3
dataset = "Standard"                                        # Test, Standard, Anomalies*
#handler = pretrain_encoders                                         

parametri = {
    "epochs": 200, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # opt               
    "mamba": False, 
    "workers": 0,
    "job_name": job_name,
    "dataset": dataset,
    "train_sar": True,                                      # train sar encoder
    "train_opt": False,                                     # train opt encoder
    "patience": 20,                                    
    "min_delta": 1e-4,
    "time_debug": False                                     # time_debug = True solo per debug, = False per training
}

print(f"PARAMETRI: {parametri}")

# volume 
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "500Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
encoders_train_func = project.new_function(
    name= f'encoders-Floods_{dataset}-{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="pretrain_encoders", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run train_encoders avviato: {run_train_encoders.id}")
print(run_train_encoders.status.state)
print(run_train_encoders.status.message)

**PLOTS**

In [ ]:
if train_sar:
    # salvataggio log
    path_s1 = project.get_artifact(f"metrics-s1_{dataset}_{job_name}").download(overwrite=True)
    df_s1 = pd.read_csv(path_s1)

    # plot
    plt.figure(figsize=(8, 5))
    plt.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
    plt.title(f'Training CAE SAR - {dataset}_{job_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.grid(True)
    plt.legend()
    plt.show()

else:
    print("train opt non eseguito")

In [ ]:
if train_opt:
    # salvataggio log
    path_s2 = project.get_artifact(f"metrics-s2_{dataset}_{job_name}").download(overwrite=True)
    df_s2 = pd.read_csv(path_s2)

    # plot
    plt.figure(figsize=(8, 5))
    plt.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
    plt.title(f'Training CAE OPT - {job_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.grid(True)
    plt.legend()
    plt.show()

else:
    print("train opt non eseguito")

In [ ]:
"""
# salvataggio log
path_s1 = project.get_artifact(f"metrics-s1_{job_name}").download(overwrite=True)
#path_s2 = project.get_artifact(f"metrics-s2_{job_name}").download(overwrite=True)

df_s1 = pd.read_csv(path_s1)
#df_s2 = pd.read_csv(path_s2)

# plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
plt.title(f"Training Encoders - {job_name}")

# sar
ax1.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
ax1.set_title(f'SAR')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

# ottico
#ax2.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
ax2.set_title(f'OTTICO')
ax2.set_xlabel('Epochs')
ax2.grid(True)

plt.show()
"""
